<a href="https://colab.research.google.com/github/molluIdontknow/ELE_Algorithm/blob/%EC%86%A1%EC%9E%AC%ED%98%84/elealgo_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 1. 캘리포니아 하우싱 데이터셋 로드
housing = fetch_california_housing(as_frame=True)
df = housing.frame
CSV_FILEPATH = "california_housing.csv"
df.to_csv(CSV_FILEPATH, index=False)

# 기본 실행 코드

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ==========================================
# 1. 데이터셋 클래스
# [강화 A] 파일 경로 대신, 이미 정규화까지 끝난 x, y numpy 배열을 직접 받도록 변경.
#          이렇게 해야 Train 데이터로만 계산한 정규화 통계(mean, std)를
#          Validation 데이터에도 "그대로" 적용할 수 있어 (Data Leakage 방지).
# ==========================================
class BatteryCSVDataset(Dataset):
    def __init__(self, x_array, y_array):
        self.x = torch.tensor(x_array, dtype=torch.float32)
        self.y = torch.tensor(y_array, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ==========================================
# MLP 모델 클래스 (기본 구조: N -> 16 -> 8 -> 1)
# [강화 B] 각 ReLU 뒤에 Dropout 추가.
#          학습 중 일부 뉴런을 무작위로 꺼서, 모델이 특정 뉴런에
#          과도하게 의존하는 것(Overfitting)을 막아줌.
# ==========================================
class SOHPredictorMLP(nn.Module):
    def __init__(self, input_dim, dropout_rate=0.2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout_rate),   # [강화 B] 추가된 줄
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Dropout(dropout_rate),   # [강화 B] 추가된 줄
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)

# 모델 실행

In [ ]:
# ==========================================
# 실행 설정
# ==========================================
CSV_FILENAME = "california_housing.csv"
FEATURES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "Latitude"]
TARGET = "MedHouseVal"

df = pd.read_csv(CSV_FILENAME)

# ==========================================
# [강화 C] Train / Validation 분리 추가
# 기존엔 전체 데이터를 그냥 다 Train으로만 썼음 -> 학습이 잘 되는지
# 확인할 "제3자 데이터"가 없었음. 이제 20%를 떼어서 Validation으로 씀.
# (나중에 진짜 배터리 데이터에선 이 부분을 Cell 단위 Split으로 교체해야 함!)
# ==========================================
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

X_train_raw = train_df[FEATURES].values
y_train = train_df[TARGET].values
X_val_raw = val_df[FEATURES].values
y_val = val_df[TARGET].values

# ==========================================
# [강화 D] Feature 정규화 추가
# Train 데이터의 평균/표준편차만 계산하고, 그 값을 Val에도 똑같이 적용.
# (Val 통계를 따로 계산해서 쓰면 안 됨 -> 이것도 일종의 Data Leakage)
# ==========================================
x_mean = X_train_raw.mean(axis=0)
x_std = X_train_raw.std(axis=0)
X_train = (X_train_raw - x_mean) / x_std
X_val = (X_val_raw - x_mean) / x_std

train_dataset = BatteryCSVDataset(X_train, y_train)
val_dataset = BatteryCSVDataset(X_val, y_val)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = SOHPredictorMLP(input_dim=len(FEATURES))
criterion = nn.MSELoss()

# ==========================================
# [강화 F] Weight Decay(L2 정규화) 추가
# Weight Decay: 가중치가 너무 커지지 않도록 억제 -> Overfitting 방지
# ==========================================
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print("\n=== 모델 학습 시작 (강화된 버전) ===")

# Validation Loss를 계산하는 함수 (학습에는 관여 안 함, 평가 전용)
def evaluate(model, loader, criterion):
    model.eval()                    # Dropout을 끄는 모드로 전환 (평가 시엔 전체 뉴런 다 사용)
    total = 0
    with torch.no_grad():           # 평가 중엔 기울기 계산 안 함 (속도/메모리 절약)
        for bx, by in loader:
            pred = model(bx)
            total += criterion(pred, by).item()
    model.train()                   # 다시 학습 모드로 전환 (Dropout 켜짐)
    return total / len(loader)

# ==========================================
# [강화 G] Early Stopping 추가
# Validation Loss가 10번 연속 최고 기록을 못 깨면 자동으로 학습을 멈추고,
# 그동안 저장해둔 "가장 좋았던 시점의 모델"을 최종적으로 사용.
# ==========================================
best_val_loss = float('inf')
patience = 10
patience_counter = 0
MAX_EPOCHS = 200

for epoch in range(1, MAX_EPOCHS + 1):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()             # 1. 기울기 초기화
        pred_y = model(batch_x)           # 2. SOH(집값) 예측
        loss = criterion(pred_y, batch_y) # 3. 오차(Loss) 계산
        loss.backward()                   # 4. 역전파
        optimizer.step()                  # 5. 가중치 업데이트

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    val_loss = evaluate(model, val_loader, criterion)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch}/{MAX_EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # --- Early Stopping 판정 ---
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")   # 지금까지 중 최고 성능 모델 저장
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEpoch {epoch}에서 Early Stopping 발동 (Val Loss {patience}번 연속 정체)")
            break

# 학습 도중 가장 좋았던 모델을 다시 불러옴 (마지막 Epoch 모델이 아님!)
model.load_state_dict(torch.load("best_model.pt"))

print(f"\n최종 Best Validation Loss: {best_val_loss:.4f}")
print("\n파이프라인 연동 테스트 완료! (강화된 버전)")
# ==========================================
# 오차율(%) 대신 MAE / RMSE / Maximum Error로 평가
# 10개 샘플이 아니라, Validation 전체에 대해 계산 (더 신뢰도 높음)
# ==========================================
model.eval()

all_actuals = []
all_preds = []

with torch.no_grad():
    for batch_x, batch_y in val_loader:
        pred = model(batch_x)
        all_actuals.append(batch_y)
        all_preds.append(pred)

all_actuals = torch.cat(all_actuals).numpy().flatten()
all_preds = torch.cat(all_preds).numpy().flatten()

abs_error = np.abs(all_actuals - all_preds)

mae = abs_error.mean()
rmse = np.sqrt(((all_actuals - all_preds) ** 2).mean())
max_error = abs_error.max()

print("\n=== 📊 Validation 전체 평가 (Notion Step 9 기준) ===")
print(f"MAE  (평균 절대 오차) : {mae:.4f}")
print(f"RMSE (평균 제곱근 오차): {rmse:.4f}")
print(f"Max Error (최대 오차)  : {max_error:.4f}")

# ==========================================
# 참고용: 10개 무작위 샘플 상세 비교표 (오차율(%) 컬럼은 제거, 절대 오차만 표시)
# ==========================================
random_indices = np.random.choice(len(all_actuals), size=10, replace=False)

result_df = pd.DataFrame({
    "실제값 (Actual)": np.round(all_actuals[random_indices], 4),
    "예측값 (Pred)": np.round(all_preds[random_indices], 4),
    "절대 오차": np.round(abs_error[random_indices], 4)
})

print("\n=== 랜덤 10개 샘플 상세 ===")
print(result_df.to_string(index=False))


=== 모델 학습 시작 (강화된 버전) ===
Epoch [1/200] | Train Loss: 1.6662 | Val Loss: 0.6911
Epoch [5/200] | Train Loss: 0.7000 | Val Loss: 0.5876
Epoch [10/200] | Train Loss: 0.6530 | Val Loss: 0.5921
Epoch [15/200] | Train Loss: 0.6298 | Val Loss: 0.5699
Epoch [20/200] | Train Loss: 0.6338 | Val Loss: 0.5634
Epoch [25/200] | Train Loss: 0.6322 | Val Loss: 0.5689
Epoch [30/200] | Train Loss: 0.6244 | Val Loss: 0.5622
Epoch [35/200] | Train Loss: 0.6271 | Val Loss: 0.5572
Epoch [40/200] | Train Loss: 0.6233 | Val Loss: 0.5538
Epoch [45/200] | Train Loss: 0.6192 | Val Loss: 0.5460
Epoch [50/200] | Train Loss: 0.6047 | Val Loss: 0.5517
Epoch [55/200] | Train Loss: 0.5964 | Val Loss: 0.5423
Epoch [60/200] | Train Loss: 0.5994 | Val Loss: 0.5480
Epoch [65/200] | Train Loss: 0.6022 | Val Loss: 0.5397
Epoch [70/200] | Train Loss: 0.5980 | Val Loss: 0.5456
Epoch [75/200] | Train Loss: 0.5930 | Val Loss: 0.5401
Epoch [80/200] | Train Loss: 0.5976 | Val Loss: 0.5404

Epoch 82에서 Early Stopping 발동 (Val Loss 